# Per-application congestion feature sensitivity — exploratory
This is an oracle-conditioned diagnostic, NOT a deployable label-dependent router and NOT a paper benchmark. Congestion levels are used as diagnostic targets. Application labels define separate analyses only.

Each capture is chronologically split into first 60% training, next 20% validation, last 20% reserved (not evaluated). Three-row trailing windows are built separately inside each partition. Training rows whose bidirectional five-tuple occurs in validation or reserved partitions are purged. Validation rows overlapping reserved tuples are also purged. Ranking and preprocessing use training only; the three selected fields are frozen before validation.

These captures were already used in earlier paper experiments: reserved rows here are not a globally unseen final test. One capture per application/level confounds congestion with capture/run conditions. Independent repeated captures are needed for causal or generalization claims. No benchmark tuning is performed here.

Rank each numeric field using Kruskal–Wallis epsilon-squared on training flow values across Low/Medium/High. This measures marginal separation, not causality or joint optimality. Select top three non-identical fields. Compare their 15 window statistics against the original timing trio using identical training/validation rows, train-fitted imputation/scaling and three-cluster MiniBatchKMeans. ARI/AMI evaluate agreement with actual congestion, independently of arbitrary cluster numbering. A 100-tree balanced RF congestion probe measures supervised separability; it is not application classification. All settings fixed before validation, seed 42.


In [ ]:
from pathlib import Path
from collections import deque
import hashlib, json, platform, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, silhouette_score)
from IPython.display import display

SEED=42
WINDOW=3
TIMING=['TcpRtt','SynAck','AckDat']
STATS=['mean','max','median','min','std']
CLASSES=['HTTP','SFTP','SMTP','SSH','Video']
MBK_CONFIG=dict(n_clusters=3,batch_size=1024,n_init=10,max_iter=100,random_state=SEED)
RF_CONFIG=dict(n_estimators=20,criterion='gini',max_depth=None,min_samples_leaf=1,
               max_features='sqrt',bootstrap=True,class_weight=None,n_jobs=-1,random_state=SEED)
RUN_REFERENCE_RF=True
DATA_DIR_OVERRIDE=None
bases=[Path.cwd(),*Path.cwd().parents]
candidates=[b/r for b in bases for r in ['CDR_MLC/DATASETS/CDR-MLC/New_Version','DATASETS/CDR-MLC/New_Version']]
DATA_DIR=Path(DATA_DIR_OVERRIDE) if DATA_DIR_OVERRIDE else next((p for p in candidates if p.is_dir()),None)
if DATA_DIR is None: raise FileNotFoundError('Set DATA_DIR_OVERRIDE')
OUT=DATA_DIR.parents[2]/'outputs'/'congestion_sensitivity'
OUT.mkdir(parents=True,exist_ok=True)
print('Data:',DATA_DIR,'\nOutput:',OUT)
print('Versions:',platform.python_version(),pd.__version__,np.__version__,sklearn.__version__)


In [ ]:
SERVICES={'HTTP':('192.168.2.122',8080),'SFTP':('192.168.2.120',22),
          'SMTP':('192.168.2.120',8025),'SSH':('192.168.2.120',22),'Video':('192.168.2.121',5000)}
CLIENT='192.168.1.111'
frames=[];audit=[]
for p in sorted(DATA_DIR.glob('*.flow')):
    m=re.fullmatch(r'(HTTP|SFTP|SMTP|SSH|Video)_(Low|Medium|High)',p.stem)
    if not m: raise ValueError(f'Unexpected capture: {p.name}')
    label,level=m.groups(); server,port=SERVICES[label]
    d=pd.read_csv(p,low_memory=False,on_bad_lines='error')
    d.columns=d.columns.str.strip()
    required=['StartTime','SrcAddr','DstAddr','Proto','Sport','Dport',*TIMING]
    if set(required)-set(d):raise ValueError(f'Missing columns in {p.name}')
    for c in d.select_dtypes('object'):d[c]=d[c].str.strip().replace('',np.nan)
    d['source_row']=np.arange(len(d))+2
    tcp=d.Proto.eq('tcp')
    forward=tcp & d.SrcAddr.eq(CLIENT) & d.DstAddr.eq(server) & pd.to_numeric(d.Dport,errors='coerce').eq(port)
    reverse=tcp & d.SrcAddr.eq(server) & d.DstAddr.eq(CLIENT) & pd.to_numeric(d.Sport,errors='coerce').eq(port)
    if reverse.any():raise ValueError(f'{p.name}: reverse-oriented records require canonicalization before analysis')
    audit.append(dict(file=p.name,raw_records=len(d),retained=int(forward.sum()),excluded=int((~forward).sum()),
                      sha256=hashlib.sha256(p.read_bytes()).hexdigest()))
    d=d.loc[forward].copy()
    if d.empty:raise ValueError(f'No service records in {p.name}')
    d['timestamp']=pd.to_datetime(d.StartTime,format='%Y/%m/%d %H:%M:%S.%f',errors='raise')
    for c in TIMING:d[c]=pd.to_numeric(d[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
    d['traffic_label']=label;d['congestion_level']=level;d['sequence_id']=p.stem;d['source_file']=p.name
    frames.append(d.sort_values(['timestamp','source_row'],kind='stable'))
data=pd.concat(frames,ignore_index=True)
assert set(zip(data.traffic_label,data.congestion_level))=={(c,l) for c in CLASSES for l in ['Low','Medium','High']}
aud=pd.DataFrame(audit);display(aud);aud.to_csv(OUT/'input_audit.csv',index=False)
display(data.groupby(['traffic_label','congestion_level']).size().unstack().reindex(columns=['Low','Medium','High']))


In [ ]:
from scipy.stats import kruskal
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score, balanced_accuracy_score
TREND_COLUMNS=[]
META={'StartTime','SrcAddr','DstAddr','Proto','Sport','Dport','Label','Cause','Dir','sTtl','dTtl',
      'traffic_label','congestion_level','sequence_id','source_file','source_row','timestamp'}
CATEGORICAL=['Flgs','State','TcpOpt']
def select_columns(train):
    nums=[];cats=[];reasons=[]
    for c in train.columns:
        reason=None
        if c in META or c in TREND_COLUMNS:reason='metadata_or_routing_statistics'
        elif re.search(r'\.\d+$',c):reason='duplicate_header'
        elif c=='IdleTime':reason='unresolved_semantics'
        elif c in CATEGORICAL:
            if train[c].nunique(dropna=True)>1:cats.append(c)
            else:reason='empty_or_constant'
        else:
            n=pd.to_numeric(train[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
            if not n.notna().any():reason='empty_or_non_numeric'
            elif n.nunique()<2:reason='constant'
            elif any(n.equals(pd.to_numeric(train[k],errors='coerce')) for k in nums):reason='identical_numeric_column'
            else:nums.append(c)
        if reason:reasons.append({'feature':c,'reason':reason})
    return nums,cats,pd.DataFrame(reasons)

LEVELS=['Low','Medium','High']
parts=[]; split_audit=[]
for seq,g in data.groupby('sequence_id',sort=False):
    g=g.sort_values(['timestamp','source_row'],kind='stable').copy()
    def key(r):
        ends=sorted([(str(r.SrcAddr),str(r.Sport)),(str(r.DstAddr),str(r.Dport))])
        return (str(r.Proto),*ends)
    g['_tuple']=[key(r) for r in g.itertuples()]
    a,b=int(.6*len(g)),int(.8*len(g))
    tr,va,hold=g.iloc[:a].copy(),g.iloc[a:b].copy(),g.iloc[b:].copy()
    tr=tr[~tr._tuple.isin(set(va._tuple)|set(hold._tuple))]
    va=va[~va._tuple.isin(set(hold._tuple))]
    for name,d in [('train',tr),('validation',va)]:
        d=d.drop(columns='_tuple');d['partition']=name;parts.append(d)
    split_audit.append(dict(sequence=seq,total=len(g),train=len(tr),validation=len(va),reserved=len(hold),purged=a+ b-a-len(tr)-len(va)))
work=pd.concat(parts,ignore_index=True)
META.add('partition')
split_audit=pd.DataFrame(split_audit);display(split_audit)
split_audit.to_csv(OUT/'split_audit.csv',index=False)

def window_matrix(frame,fields):
    blocks=[]
    for seq,g in frame.groupby('sequence_id',sort=False):
        vals=g[fields].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
        # Full chronological windows only. Missing measurements are imputed later using training medians.
        out={}
        for f in fields:
            r=vals[f].rolling(3,min_periods=1)
            for stat in STATS:
                out[f+'_'+stat]=r.std(ddof=0) if stat=='std' else getattr(r,stat)()
        z=pd.DataFrame(out,index=g.index).iloc[2:]
        blocks.append(z)
    return pd.concat(blocks)

rankings=[]; comparisons=[]; selected_rows=[]
for label in CLASSES:
    tr=work[(work.traffic_label==label)&(work.partition=='train')].copy()
    va=work[(work.traffic_label==label)&(work.partition=='validation')].copy()
    nums,_,excluded=select_columns(tr)
    excluded.to_csv(OUT/(label+'_excluded.csv'),index=False)
    scores=[]
    for f in nums:
        groups=[pd.to_numeric(tr.loc[tr.congestion_level==lev,f],errors='coerce').replace([np.inf,-np.inf],np.nan).dropna().values for lev in LEVELS]
        if min(map(len,groups))<10:continue
        try:h=float(kruskal(*groups).statistic)
        except ValueError:continue
        n=sum(map(len,groups));effect=max(0.,(h-2)/(n-3))
        scores.append(dict(label=label,feature=f,train_epsilon2=effect,n=n))
    rank=pd.DataFrame(scores).sort_values(['train_epsilon2','feature'],ascending=[False,True])
    chosen=rank.feature.head(3).tolist()
    if set(chosen)==set(TIMING):chosen=list(TIMING)  # identical sets must retain identical column order
    if len(chosen)!=3:raise ValueError('Insufficient features')
    rankings.append(rank);selected_rows.append(dict(label=label,selected=', '.join(chosen)))
    for name,fields in [('original_timing',TIMING),('selected_three',chosen)]:
        x=window_matrix(tr,fields);v=window_matrix(va,fields)
        yt=tr.loc[x.index,'congestion_level'];yv=va.loc[v.index,'congestion_level']
        assert set(yt)==set(LEVELS) and set(yv)==set(LEVELS)
        pre=Pipeline([('impute',SimpleImputer(strategy='median',keep_empty_features=True)),('scale',StandardScaler())])
        xx=pre.fit_transform(x);vv=pre.transform(v)
        model=MiniBatchKMeans(**MBK_CONFIG).fit(xx)
        routes=model.predict(vv)
        probe=RandomForestClassifier(n_estimators=100,class_weight='balanced',random_state=SEED,n_jobs=-1).fit(xx,yt)
        comparisons.append(dict(label=label,features=name,train_n=len(x),validation_n=len(v),ARI=adjusted_rand_score(yv,routes),AMI=adjusted_mutual_info_score(yv,routes),congestion_balanced_accuracy=balanced_accuracy_score(yv,probe.predict(vv))))
        pd.crosstab(yv,pd.Series(routes,index=v.index,name='cluster')).to_csv(OUT/(label+'_'+name+'_validation_clusters.csv'))
rankings=pd.concat(rankings,ignore_index=True);comparisons=pd.DataFrame(comparisons);selected=pd.DataFrame(selected_rows)
rankings.to_csv(OUT/'feature_ranking.csv',index=False);comparisons.to_csv(OUT/'comparison.csv',index=False);selected.to_csv(OUT/'selected.csv',index=False)
display(selected);display(comparisons)


## Interpretation
ARI/AMI near zero indicate little agreement with the collected congestion levels, even when clusters contain many HTTP records. A high supervised probe score with weak ARI means features contain level information but MiniBatchKMeans does not recover the levels well. Gains on validation are exploratory and do not justify routing using unknown true application labels. Selection across all three levels is incompatible with claiming a Low-only trained paper scenario; keep this experiment separate.


## Executed validation results
All three code cells completed. Training-only selection gave HTTP: TcpRtt, AckDat, SrcLoad; SMTP: SrcRate, DstLoad, DstRate; SFTP/SSH/Video: the original timing trio. Validation ARI (original → selected): HTTP 0.381235 → 0.430133; SFTP 0.281957 → 0.281957; SMTP 0.090335 → 0.025512; SSH 0.352673 → 0.352673; Video 0.296084 → 0.296084.

These are congestion-cluster agreement scores, not application classification accuracy. HTTP shows a modest benefit under this split. Selection did not help Video and did not generalize for SMTP. This is evidence against adopting label-dependent feature selection as a blanket solution. No final reserved partition was evaluated. Run All regenerates detailed outputs.
